# Football Player Market Value Prediction & Scouting Dashboard
## Notebook 1 — Data Collection

**Author:** Ashish Abraham  
**Date:** July 2026  
**Description:** Scrapes player performance statistics from FBref across the Big 5 European leagues for 5 seasons (2020-21 to 2024-25). Data saved locally as CSV files.

**Stat Categories Collected:** shooting, misc, passing, defense, possession, gca, passing_types

**Note:** standard, keeper, and playing_time stat pages load via JavaScript and cannot be scraped. xG is noted as a limitation. All other required features are covered across the 7 working stat types.

In [21]:
import soccerdata as sd
import pandas as pd
from io import BytesIO
import time
import os

# Initialise FBref browser
fbref = sd.FBref("Big 5 European Leagues Combined", ["2425"])

save_path = "../data/raw/fbref/"

# 7 working stat types only
all_stats = [
    "shooting", "misc", "passing", "defense",
    "possession", "gca", "passing_types"
]

# All 5 seasons
seasons = {
    "2021": "2020-2021",
    "2122": "2021-2022",
    "2223": "2022-2023",
    "2324": "2023-2024",
    "2425": "2024-2025",
    "2526": ""
}

def scrape_fbref_big5(stat_type, season_code, season_url_part):
    if season_code == "2425":
        url = f"https://fbref.com/en/comps/Big5/{stat_type}/players/Big-5-European-Leagues-Stats"
    else:
        url = f"https://fbref.com/en/comps/Big5/{season_url_part}/{stat_type}/players/{season_url_part}-Big-5-European-Leagues-Stats"
    
    response = fbref.get(url)
    content = response.read()
    tables = pd.read_html(BytesIO(content))
    df = tables[-1]  # Player stats table is always last
    df.columns = ['_'.join(col).strip('_') for col in df.columns]
    df['season'] = season_code
    return df

print("Setup complete — ready to scrape")

[07/02/26 21:41:20] INFO     Saving cached data to /Users/ashish/soccerdata/data/FBref               _common.py:250

Setup complete — ready to scrape


## Step 1 — Scrape All Stat Types Across All 5 Seasons

Pulls 7 stat categories across 5 seasons. Allow 40-50 minutes.
Each file saved as `fbref_{stat}_{season}.csv` in `data/raw/fbref/`.

In [18]:
for season_code, season_url in seasons.items():
    print(f"\n=== Season: {season_code} ===")
    for stat in all_stats:
        print(f"  Pulling {stat}...")
        df = scrape_fbref_big5(stat, season_code, season_url)
        filename = f"fbref_{stat}_{season_code}.csv"
        df.to_csv(os.path.join(save_path, filename), index=False)
        print(f"  Saved {filename} — shape: {df.shape}")
        time.sleep(4)

print("\nAll data scraped and saved successfully!")


=== Season: 2021 ===
  Pulling shooting...
  Saved fbref_shooting_2021.csv — shape: (2934, 21)
  Pulling misc...
  Saved fbref_misc_2021.csv — shape: (2934, 23)
  Pulling passing...
  Saved fbref_passing_2021.csv — shape: (2934, 31)
  Pulling defense...
  Saved fbref_defense_2021.csv — shape: (2934, 27)
  Pulling possession...
  Saved fbref_possession_2021.csv — shape: (5, 4)
  Pulling gca...
  Saved fbref_gca_2021.csv — shape: (2934, 27)
  Pulling passing_types...
  Saved fbref_passing_types_2021.csv — shape: (2934, 26)

=== Season: 2122 ===
  Pulling shooting...
  Saved fbref_shooting_2122.csv — shape: (3038, 21)
  Pulling misc...
  Saved fbref_misc_2122.csv — shape: (3038, 23)
  Pulling passing...
  Saved fbref_passing_2122.csv — shape: (3038, 31)
  Pulling defense...
  Saved fbref_defense_2122.csv — shape: (3038, 27)
  Pulling possession...
  Saved fbref_possession_2122.csv — shape: (3038, 31)
  Pulling gca...
  Saved fbref_gca_2122.csv — shape: (3038, 27)
  Pulling passing_types.

In [19]:
print("Re-scraping possession 2021...")
df = scrape_fbref_big5("possession", "2021", "2020-2021")
df.to_csv(os.path.join(save_path, "fbref_possession_2021.csv"), index=False)
print(f"Fixed — shape: {df.shape}")

Re-scraping possession 2021...
Fixed — shape: (2934, 31)


In [20]:
print("=== Adding Season: 2526 (2025-2026) ===")
for stat in all_stats:
    print(f"  Pulling {stat}...")
    df = scrape_fbref_big5(stat, "2526", "")
    filename = f"fbref_{stat}_2526.csv"
    df.to_csv(os.path.join(save_path, filename), index=False)
    print(f"  Saved {filename} — shape: {df.shape}")
    time.sleep(4)

print("\n2025-26 season added!")

=== Adding Season: 2526 (2025-2026) ===
  Pulling shooting...
  Saved fbref_shooting_2526.csv — shape: (2952, 21)
  Pulling misc...
  Saved fbref_misc_2526.csv — shape: (2952, 23)
  Pulling passing...
  Saved fbref_passing_2526.csv — shape: (2952, 31)
  Pulling defense...
  Saved fbref_defense_2526.csv — shape: (2952, 27)
  Pulling possession...
  Saved fbref_possession_2526.csv — shape: (2952, 31)
  Pulling gca...
  Saved fbref_gca_2526.csv — shape: (2952, 27)
  Pulling passing_types...
  Saved fbref_passing_types_2526.csv — shape: (2952, 26)

2025-26 season added!


## Step 2 — Combine Individual Season Files Into All-Seasons Files

Concatenates all 5 season files per stat type into a single all_seasons file.
Deletes individual season files after combining.

In [22]:
print("Combining season files...")

for stat in all_stats:
    dfs = []
    for season_code in seasons.keys():
        filepath = f"{save_path}fbref_{stat}_{season_code}.csv"
        df = pd.read_csv(filepath)
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    combined.to_csv(f"{save_path}fbref_{stat}_all_seasons.csv", index=False)
    print(f"  fbref_{stat}_all_seasons.csv — shape: {combined.shape}")

print("\nAll combined files created!")

Combining season files...
  fbref_shooting_all_seasons.csv — shape: (17846, 21)
  fbref_misc_all_seasons.csv — shape: (17846, 23)
  fbref_passing_all_seasons.csv — shape: (17846, 31)
  fbref_defense_all_seasons.csv — shape: (17846, 27)
  fbref_possession_all_seasons.csv — shape: (17846, 31)
  fbref_gca_all_seasons.csv — shape: (17846, 27)
  fbref_passing_types_all_seasons.csv — shape: (17846, 26)

All combined files created!


## Step 3 — Verify Final Files

In [23]:
print("=== FINAL FILE INVENTORY ===\n")

files = sorted([f for f in os.listdir(save_path) if f.endswith("_all_seasons.csv")])

for f in files:
    df = pd.read_csv(os.path.join(save_path, f))
    print(f"{f}")
    print(f"  Shape: {df.shape}")
    print(f"  Seasons: {sorted(df['season'].unique().tolist())}")
    print(f"  Sample players: {df['Unnamed: 1_level_0_Player'].dropna().head(3).tolist()}")
    print()

print(f"Total stat files: {len(files)}")
print("Data collection complete.")

=== FINAL FILE INVENTORY ===

fbref_defense_all_seasons.csv
  Shape: (17846, 27)
  Seasons: [2021, 2122, 2223, 2324, 2425, 2526]
  Sample players: ['Ismael Aaneba', 'Patrick van Aanholt', 'Yunis Abdelhamid']

fbref_gca_all_seasons.csv
  Shape: (17846, 27)
  Seasons: [2021, 2122, 2223, 2324, 2425, 2526]
  Sample players: ['Ismael Aaneba', 'Patrick van Aanholt', 'Yunis Abdelhamid']

fbref_misc_all_seasons.csv
  Shape: (17846, 23)
  Seasons: [2021, 2122, 2223, 2324, 2425, 2526]
  Sample players: ['Ismael Aaneba', 'Patrick van Aanholt', 'Yunis Abdelhamid']

fbref_passing_all_seasons.csv
  Shape: (17846, 31)
  Seasons: [2021, 2122, 2223, 2324, 2425, 2526]
  Sample players: ['Ismael Aaneba', 'Patrick van Aanholt', 'Yunis Abdelhamid']

fbref_passing_types_all_seasons.csv
  Shape: (17846, 26)
  Seasons: [2021, 2122, 2223, 2324, 2425, 2526]
  Sample players: ['Ismael Aaneba', 'Patrick van Aanholt', 'Yunis Abdelhamid']

fbref_possession_all_seasons.csv
  Shape: (17846, 31)
  Seasons: [2021, 2122